In [1]:
%load_ext rpy2.ipython

In [2]:
x = [1, 2, 3, 4, 5]
y = [2, 4, 6, 8, 10]

In [3]:
%%R -i x,y -o slope,intercept

model <- lm(y ~ x)
slope <- coef(model)[2]
intercept <- coef(model)[1]
print(paste("The slope is", slope, "and the intercept is", intercept))

[1] "The slope is 2 and the intercept is 0"


In [6]:
%%R

#load library
library(yogiroc)
library(data.table)

# -------------------------------------------------------------------------
# 1) Obtain scores and reference set (annotated patho. and bengin variants)
# -------------------------------------------------------------------------

FunctionalScores<-fread("SOD1.csv")
FunctionalScores <- FunctionalScores[, .(hgvs_pro, score)]
#FunctionalScores$duplicate<-FunctionalScores$score # Example of additional dataset
FunctionalScores$score <- FunctionalScores$score *-1
#FunctionalScores$duplicate <- FunctionalScores$duplicate*-1

ReferenceSet<-fread("referenceSet.csv")
ReferenceSet[, Pathogenic := ifelse(referenceSet == "Negative", FALSE, TRUE)]

prcdata<-merge(FunctionalScores, ReferenceSet, by = "hgvs_pro")
prcdata <- prcdata[, .(Pathogenic, score)]
prcdata<-na.omit(prcdata)

names(prcdata)[2] <- "SOD1" 
#names(prcdata)[3] <- "SOD1"  #Add curve for additional dataset like this

#create yogiroc objects for the predictions in the data
yrobj <- yr2(truth=prcdata$Pathogenic,scores=prcdata[,-1])

#Automatically saves plot into folder with indicated name
pdf(file="test.pdf", width=5.5, height=5.5)

#draw the PRC curve(s)
draw.prc(yrobj, balanced=TRUE, main="")

abline(h=90, lty="dashed")

dev.off()

png 
  2 
